# Arbigen · Cuaderno de ciencia de datos

Este cuaderno analiza **una investigación exportada desde Arbigen**. Solo necesita el JSON descargado con «Descargar datos para el cuaderno» y Python 3.10 o posterior; funciona en Google Colab o en un entorno local sin instalar paquetes, clonar el repositorio ni conectar a la API, PostgreSQL o Apify. En Colab, sube el JSON al panel de archivos y ejecuta «Entorno de ejecución → Ejecutar todo». En local, guárdalo junto al cuaderno o en `notebooks/data/`.

Las reglas de ETL, variables, segmentación, pronóstico, rentabilidad y Opportunity Score están **incluidas como una captura autocontenida** del backend en la revisión `2299aeb`. Así los cálculos de este cuaderno corresponden a esa versión; si Arbigen cambia sus modelos, hay que actualizar esta captura antes de comparar resultados nuevos. Un segundo JSON de escenario financiero es opcional: si lo tienes, define `ARBIGEN_SCENARIO_PATH` con su ruta. No se inventan costos ni se ejecutan búsquedas pagadas.

Las celdas de reglas incluidas se pueden desplegar para revisar el código, pero no necesitan editarse. Ejecuta el cuaderno de arriba abajo.


## 1. Procedencia y límites de las fuentes

1. Abre una investigación en Arbigen y pulsa **«Descargar datos para el cuaderno»**. No hace falta repetir una búsqueda guardada.
2. En Colab, sube `arbigen-dataset-*.json` al panel de archivos. En local, colócalo junto al cuaderno o en `notebooks/data/`. Si hay varios archivos, se usa el más reciente; `ARBIGEN_DATASET_PATH` permite elegir uno explícitamente.
3. La muestra contiene publicaciones observadas, no ventas ni cuota de mercado. Trends es un índice relativo 0–100 y `<1` es un dato censurado, no cero. Cada país conserva su moneda.

El JSON contiene campos internos normalizados y la procedencia de Trends, sin el token de Apify. Si `trend_source` es `google_trends_csv_geo_unverified`, el CSV no confirma su geografía: verifica el país antes de interpretar resultados. Evita publicar el JSON si sus términos o datos son sensibles.

La exportación ampliada incluye `snapshot.reported_total_results` y `snapshot.items[].signals` con datos opcionales de precio anterior, envío, tienda, reseñas, ventas declaradas, categoría y otros. En archivos anteriores las señales ausentes aparecerán como «no informado».


In [ ]:
import json
import os
from pathlib import Path
from datetime import date
from decimal import Decimal

# En Colab, sube el JSON de Arbigen al panel de archivos (/content).
# En local, guárdalo junto al cuaderno o en notebooks/data/.
cwd = Path.cwd().resolve()
search_dirs = (cwd, cwd / "data", cwd / "notebooks" / "data")
choices = [path for folder in search_dirs for path in folder.glob("arbigen-dataset-*.json")]
choices += [folder / "dataset.json" for folder in search_dirs if (folder / "dataset.json").is_file()]
explicit_path = os.environ.get("ARBIGEN_DATASET_PATH")
path = Path(explicit_path).expanduser() if explicit_path else max(choices, key=lambda file: file.stat().st_mtime) if choices else None
if path is None or not path.is_file():
    raise FileNotFoundError("Sube arbigen-dataset-*.json a Colab o colócalo en notebooks/data; también puedes definir ARBIGEN_DATASET_PATH.")
data = json.loads(path.read_text(encoding="utf-8"))
if data.get("country") not in {"MX", "CO", "AR"}:
    raise ValueError("El JSON no es una exportación de Arbigen para MX, CO o AR.")
print(f"Archivo: {path.name} · país: {data['country']} · muestra: {len((data.get('snapshot') or {}).get('items', []))} productos · Trends: {len(data.get('trends', []))} puntos")
print("Procedencia Trends:", data.get("trend_source") or "sin serie", "· importado:", data.get("trend_imported_at") or "—")
if data.get("trend_source") == "google_trends_csv_geo_unverified":
    print("Advertencia: el CSV no declara la geografía; verifica manualmente el país de exportación.")


## 2. ETL reproducible

El ETL normaliza títulos, elimina duplicados por ID y conserva para el análisis de precios solo productos con moneda y precio válidos. Las pistas de material salen del título y **no son atributos verificados**. Si el archivo no contiene muestra, el informe de calidad de productos queda sin procedencia; los puntos de Trends se pueden analizar igualmente.


In [ ]:
# Referencia: backend/app/modules/marketplace/domain.py (2299aeb)
from dataclasses import dataclass, field
from datetime import datetime
from decimal import Decimal
from typing import Literal, Protocol


@dataclass(frozen=True)
class ListingSignals:
    """Optional listing observations; None means the source did not report a value."""

    previous_price: Decimal | None = None
    free_shipping: bool | None = None
    official_store: bool | None = None
    international_purchase: bool | None = None
    stock_available: bool | None = None
    sold_quantity: int | None = None
    review_count: int | None = None
    rating: Decimal | None = None
    position: int | None = None
    seller_id: str | None = None
    brand: str | None = None
    category_id: str | None = None
    domain_id: str | None = None
    catalog_product_id: str | None = None
    variation_id: str | None = None


@dataclass(frozen=True)
class MarketplaceProduct:
    id: str
    title: str
    price: Decimal | None
    currency: str | None
    permalink: str | None
    image_url: str | None
    signals: ListingSignals = field(default_factory=ListingSignals)


@dataclass(frozen=True)
class MarketplaceSearchResult:
    source: Literal["listings", "catalog"]
    site_id: str
    query: str
    fetched_at: datetime
    items: list[MarketplaceProduct]
    reported_total_results: int | None = None


class MarketplaceSearchProvider(Protocol):
    def search(self, *, country: str, query: str, limit: int) -> MarketplaceSearchResult: ...


class MarketplaceSearchError(Exception):
    def __init__(self, code: str):
        self.code = code
        super().__init__(code)


# Referencia: backend/app/modules/trends/domain.py (2299aeb)
from dataclasses import dataclass
from datetime import date
from decimal import Decimal
from typing import Protocol


@dataclass(frozen=True)
class TrendObservation:
    date: date
    value: Decimal | None
    less_than_one: bool = False


@dataclass(frozen=True)
class TrendSeries:
    source: str
    points: tuple[TrendObservation, ...]


class TrendsInputError(ValueError):
    pass


class TrendsProvider(Protocol):
    def load(self, *, query: str, country: str, payload: bytes) -> TrendSeries: ...


# Referencia: backend/app/modules/etl/domain.py (2299aeb)
from dataclasses import dataclass, field
from datetime import date
from decimal import Decimal



@dataclass(frozen=True)
class AnalyticalProduct:
    external_id: str
    title: str
    price: Decimal
    currency: str
    permalink: str | None
    image_url: str | None
    attributes: dict[str, str]
    signals: ListingSignals = field(default_factory=ListingSignals)


@dataclass(frozen=True)
class AnalyticalTrendPoint:
    date: date
    value: Decimal | None
    less_than_one: bool


@dataclass(frozen=True)
class ProductQuality:
    source_count: int
    duplicate_count: int
    missing_title_count: int
    missing_price_count: int
    invalid_price_count: int
    invalid_currency_count: int
    included_count: int


@dataclass(frozen=True)
class PreparedDataset:
    products: tuple[AnalyticalProduct, ...]
    trends: tuple[AnalyticalTrendPoint, ...]
    product_quality: ProductQuality


# Referencia: backend/app/modules/etl/pipeline.py (2299aeb)
"""Reglas ETL sobre contratos internos, sin referencias a proveedores externos."""

import html
import re
import unicodedata
from datetime import date
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from typing import Iterable



COUNTRY_CURRENCY = {"MX": "MXN", "CO": "COP", "AR": "ARS"}
MATERIAL_HINTS = ("acero inoxidable", "algodon", "madera", "cuero", "plata")
MAX_PRICE = Decimal("9999999999.99")


def _title(value: str) -> str:
    return " ".join(unicodedata.normalize("NFKC", html.unescape(value)).split())[:300]


def _fold(value: str) -> str:
    return "".join(character for character in unicodedata.normalize("NFKD", value.casefold()) if not unicodedata.combining(character))


def _attributes(title: str) -> dict[str, str]:
    folded = _fold(title)
    for material in MATERIAL_HINTS:
        if re.search(rf"(?<!\w){re.escape(material)}(?!\w)", folded):
            return {"material_hint": material, "hint_source": "title"}
    return {}


def _price(value: Decimal | None) -> Decimal | None:
    if value is None:
        return None
    try:
        if not value.is_finite() or value < 0 or value > MAX_PRICE:
            return None
        rounded = value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    except (InvalidOperation, AttributeError):
        return None
    return rounded if rounded <= MAX_PRICE else None


def _candidate_rank(product: MarketplaceProduct, expected_currency: str) -> tuple[bool, bool, bool, bool]:
    valid_price = _price(product.price) is not None
    return (
        bool(_title(product.title)) and valid_price and product.currency == expected_currency,
        bool(_title(product.title)),
        valid_price and product.currency == expected_currency,
        bool(product.permalink),
    )


def prepare_trends(trends: Iterable[TrendObservation]) -> tuple[AnalyticalTrendPoint, ...]:
    trend_rows: dict[date, AnalyticalTrendPoint] = {}
    for point in trends:
        if point.date in trend_rows:
            raise ValueError("La serie de Trends contiene fechas duplicadas.")
        if point.less_than_one:
            if point.value is not None:
                raise ValueError("Un valor censurado de Trends no debe tener índice numérico.")
        elif point.value is None or not point.value.is_finite() or not 0 <= point.value <= 100:
            raise ValueError("La serie de Trends contiene un índice inválido.")
        trend_rows[point.date] = AnalyticalTrendPoint(point.date, point.value, point.less_than_one)
    return tuple(trend_rows[key] for key in sorted(trend_rows))


def prepare_dataset(*, country: str, products: Iterable[MarketplaceProduct], trends: Iterable[TrendObservation]) -> PreparedDataset:
    if country not in COUNTRY_CURRENCY:
        raise ValueError("País no admitido para el dataset analítico.")
    expected_currency = COUNTRY_CURRENCY[country]
    rows = list(products)
    selected: dict[str, MarketplaceProduct] = {}
    duplicates = 0
    for product in rows:
        if product.id in selected:
            duplicates += 1
            previous = selected[product.id]
            if _candidate_rank(product, expected_currency) > _candidate_rank(previous, expected_currency):
                selected[product.id] = product
        else:
            selected[product.id] = product

    clean: list[AnalyticalProduct] = []
    missing_title = missing_price = invalid_price = invalid_currency = 0
    for product in selected.values():
        title = _title(product.title)
        if not title:
            missing_title += 1
            continue
        if product.price is None:
            missing_price += 1
            continue
        price = _price(product.price)
        if price is None:
            invalid_price += 1
            continue
        if product.currency != expected_currency:
            invalid_currency += 1
            continue
        clean.append(AnalyticalProduct(product.id, title, price, expected_currency, product.permalink, product.image_url, _attributes(title), product.signals))

    quality = ProductQuality(len(rows), duplicates, missing_title, missing_price, invalid_price, invalid_currency, len(clean))
    return PreparedDataset(tuple(clean), prepare_trends(trends), quality)


In [ ]:
snapshot = data.get("snapshot")
raw_items = (snapshot or {}).get("items", [])
products = [MarketplaceProduct(
    id=row["id"], title=row["title"],
    price=Decimal(str(row["price"])) if row.get("price") is not None else None,
    currency=row.get("currency"), permalink=row.get("permalink"), image_url=row.get("image_url"),
) for row in raw_items]
observations = [TrendObservation(
    date=date.fromisoformat(row["date"]),
    value=Decimal(str(row["value"])) if row.get("value") is not None else None,
    less_than_one=row["less_than_one"],
) for row in data.get("trends", [])]
prepared = prepare_dataset(country=data["country"], products=products, trends=observations)
quality = prepared.product_quality
print("Entradas:", quality.source_count, "· depurados con precio:", quality.included_count)
print("Duplicados:", quality.duplicate_count, "· sin precio:", quality.missing_price_count)
print("Precio inválido:", quality.invalid_price_count, "· moneda inválida:", quality.invalid_currency_count)
print("Puntos de Trends:", len(prepared.trends), "· censurados <1:", sum(point.less_than_one for point in prepared.trends))


### 2.1 Señales adicionales de los anuncios

Estas consultas usan únicamente `snapshot.items[].signals` del JSON exportado. La cobertura se calcula antes de resumir cada variable; `null` y campos inexistentes significan **no informado**. Los conteos de ventas y reseñas, cuando aparecen, describen lo mostrado por el anuncio y no ventas verificadas. `reported_total_results` es un contexto de la búsqueda, no el tamaño de la muestra ni un censo de competidores.

In [ ]:
from collections import Counter
from statistics import median

signal_names = ("previous_price", "free_shipping", "official_store", "international_purchase",
                "stock_available", "sold_quantity", "review_count", "rating", "position",
                "seller_id", "brand", "category_id", "domain_id", "catalog_product_id", "variation_id")
signals = [row.get("signals") or {} for row in raw_items]
coverage = {name: sum(record.get(name) is not None for record in signals) for name in signal_names}
print("Resultados reportados por la fuente:", (snapshot or {}).get("reported_total_results") or "No informados")
print("Cobertura por campo · anuncios:", len(raw_items))
for name, count in coverage.items():
    print(f"{name:<25} {count:>3}/{len(raw_items):<3} ({100 * count / len(raw_items) if raw_items else 0:5.1f} %)")
for name in ("category_id", "domain_id", "brand", "seller_id"):
    counts = Counter(record[name] for record in signals if record.get(name) is not None)
    print(name, "más frecuentes:", counts.most_common(5) or "Sin datos")
for name in ("free_shipping", "official_store", "stock_available"):
    known = [record[name] for record in signals if isinstance(record.get(name), bool)]
    print(name, f"{sum(known)}/{len(known)} positivos entre informados" if known else "Sin datos")
for name in ("sold_quantity", "review_count", "rating"):
    known = [Decimal(str(record[name])) for record in signals if record.get(name) is not None]
    print(name, "mediana:", median(known) if known else "No disponible")
discounts = []
for row in raw_items:
    current, previous = row.get("price"), (row.get("signals") or {}).get("previous_price")
    if current is not None and previous is not None:
        current, previous = Decimal(str(current)), Decimal(str(previous))
        if previous > current and previous > 0:
            discounts.append((previous - current) / previous * 100)
print("Descuento aparente mediano:", round(median(discounts), 2) if discounts else "No disponible")

## 3. Ingeniería de variables

Las estadísticas de precio se calculan sobre productos depurados y en una sola moneda. La desviación estándar es **poblacional**. La cobertura usa el número de entradas de la muestra como denominador. Para Trends, cada ventana incluye tres observaciones consecutivas a frecuencia regular (diaria, semanal o mensual); no se imputan valores `<1`. Si faltan datos, la variable queda como `None`.


In [ ]:
# Referencia: backend/app/modules/features/domain.py (2299aeb)
from dataclasses import dataclass
from decimal import Decimal


@dataclass(frozen=True)
class MarketFeatures:
    currency: str
    source_count: int | None
    priced_count: int
    price_coverage_pct: Decimal | None
    price_min: Decimal | None
    price_max: Decimal | None
    price_mean: Decimal | None
    price_median: Decimal | None
    price_stddev: Decimal | None
    material_hints: dict[str, int]


@dataclass(frozen=True)
class TrendFeatures:
    point_count: int
    observed_count: int
    censored_count: int
    observed_coverage_pct: Decimal | None
    mean_index: Decimal | None
    volatility_index: Decimal | None
    moving_average_3: Decimal | None
    growth_3v3_pct: Decimal | None
    momentum_3v3: Decimal | None
    acceleration_3v3: Decimal | None


@dataclass(frozen=True)
class FeatureReport:
    country: str
    market: MarketFeatures
    trend: TrendFeatures


# Referencia: backend/app/modules/features/engineering.py (2299aeb)
"""Variables descriptivas puras; no consulta proveedores ni estima ventas."""

from collections import Counter
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from typing import Sequence



HUNDRED = Decimal(100)
TWO_DECIMALS = Decimal("0.01")


def _rounded(value: Decimal) -> Decimal:
    return value.quantize(TWO_DECIMALS, rounding=ROUND_HALF_UP)


def _mean(values: Sequence[Decimal]) -> Decimal:
    return sum(values, Decimal(0)) / len(values)


def _median(values: Sequence[Decimal]) -> Decimal:
    ordered = sorted(values)
    center = len(ordered) // 2
    return ordered[center] if len(ordered) % 2 else (ordered[center - 1] + ordered[center]) / 2


def _stddev(values: Sequence[Decimal]) -> Decimal:
    average = _mean(values)
    return (_mean([(value - average) ** 2 for value in values])).sqrt()


def _regular_dates(dates: Sequence[date]) -> bool:
    gaps = [(right - left).days for left, right in zip(dates, dates[1:])]
    return bool(gaps) and (all(gap == 1 for gap in gaps) or all(gap == 7 for gap in gaps) or all(28 <= gap <= 31 for gap in gaps))


def _window_means(points: Sequence[AnalyticalTrendPoint], window_count: int) -> list[Decimal] | None:
    required = window_count * 3
    if len(points) < required:
        return None
    recent = points[-required:]
    if any(point.value is None or point.less_than_one for point in recent) or not _regular_dates([point.date for point in recent]):
        return None
    return [_mean([point.value for point in recent[index:index + 3] if point.value is not None]) for index in range(0, required, 3)]


def calculate_features(
    *, country: str, products: Sequence[AnalyticalProduct],
    trends: Sequence[AnalyticalTrendPoint], source_count: int | None,
) -> FeatureReport:
    if country not in COUNTRY_CURRENCY:
        raise ValueError("País no admitido para ingeniería de variables.")
    if source_count is not None and (source_count < 0 or source_count < len(products)):
        raise ValueError("El conteo de entrada no es compatible con los productos depurados.")
    currency = COUNTRY_CURRENCY[country]
    if any(product.currency != currency for product in products):
        raise ValueError("El dataset contiene monedas distintas a la del país.")
    prices = [product.price for product in products]
    material_hints = dict(sorted(Counter(product.attributes["material_hint"] for product in products if product.attributes.get("material_hint")).items()))
    market = MarketFeatures(
        currency=currency,
        source_count=source_count,
        priced_count=len(prices),
        price_coverage_pct=_rounded(Decimal(len(prices)) / source_count * HUNDRED) if source_count else None,
        price_min=min(prices) if prices else None,
        price_max=max(prices) if prices else None,
        price_mean=_rounded(_mean(prices)) if prices else None,
        price_median=_rounded(_median(prices)) if prices else None,
        price_stddev=_rounded(_stddev(prices)) if len(prices) >= 2 else None,
        material_hints=material_hints,
    )

    ordered = sorted(trends, key=lambda point: point.date)
    observed = [point.value for point in ordered if point.value is not None and not point.less_than_one]
    censored = sum(point.less_than_one for point in ordered)
    last_window = _window_means(ordered, 1)
    two_windows = _window_means(ordered, 2)
    three_windows = _window_means(ordered, 3)
    trend = TrendFeatures(
        point_count=len(ordered),
        observed_count=len(observed),
        censored_count=censored,
        observed_coverage_pct=_rounded(Decimal(len(observed)) / len(ordered) * HUNDRED) if ordered else None,
        mean_index=_rounded(_mean(observed)) if observed else None,
        volatility_index=_rounded(_stddev(observed)) if len(observed) >= 2 else None,
        moving_average_3=_rounded(last_window[0]) if last_window else None,
        growth_3v3_pct=_rounded((two_windows[1] - two_windows[0]) / two_windows[0] * HUNDRED) if two_windows and two_windows[0] > 0 else None,
        momentum_3v3=_rounded(two_windows[1] - two_windows[0]) if two_windows else None,
        acceleration_3v3=_rounded(three_windows[2] - 2 * three_windows[1] + three_windows[0]) if three_windows else None,
    )
    return FeatureReport(country, market, trend)


In [ ]:
features = calculate_features(
    country=data["country"], products=prepared.products, trends=prepared.trends,
    source_count=len(raw_items) if snapshot is not None else None,
)

def display_table(title, rows):
    print("\n" + title)
    print("-" * len(title))
    for label, value in rows:
        print(f"{label:<32} {value if value is not None else 'No disponible'}")

m, t = features.market, features.trend
display_table("MERCADO", [
    ("Moneda", m.currency), ("Entradas de la muestra", m.source_count),
    ("Productos con precio", m.priced_count), ("Cobertura de precios (%)", m.price_coverage_pct),
    ("Precio mínimo", m.price_min), ("Precio mediano", m.price_median),
    ("Precio medio", m.price_mean), ("Precio máximo", m.price_max),
    ("Desviación estándar de precio", m.price_stddev),
    ("Pistas de material", m.material_hints or "No disponibles"),
])
display_table("TRENDS · ÍNDICE RELATIVO", [
    ("Puntos totales", t.point_count), ("Puntos observados", t.observed_count),
    ("Puntos censurados <1", t.censored_count), ("Cobertura observada (%)", t.observed_coverage_pct),
    ("Índice medio", t.mean_index), ("Volatilidad del índice", t.volatility_index),
    ("Media móvil de 3", t.moving_average_3), ("Crecimiento 3 vs. 3 (%)", t.growth_3v3_pct),
    ("Momentum 3 vs. 3", t.momentum_3v3), ("Aceleración 3 vs. 3", t.acceleration_3v3),
])


### 3.1 Distribución de precios de la muestra

La ficha de resultados muestra hasta cinco rangos de precio de la muestra depurada. El rango completo entre el mínimo y el máximo se divide en partes de igual ancho; el extremo máximo entra en la última barra. Si todos los productos tienen el mismo precio, se usa una sola barra. Esta visualización cuenta publicaciones observadas, **no ventas ni demanda**. Las cantidades deben sumar el número de productos con precio válido.


In [ ]:
from math import floor

prices = [float(product.price) for product in prepared.products if product.price > 0]
if prices:
    lowest, highest = min(prices), max(prices)
    number_of_bins = 1 if lowest == highest else min(5, len(set(prices)))
    width = 0 if number_of_bins == 1 else (highest - lowest) / number_of_bins
    counts = [0] * number_of_bins
    for price in prices:
        position = 0 if width == 0 else min(number_of_bins - 1, floor((price - lowest) / width))
        counts[position] += 1

    for index, count in enumerate(counts):
        lower = lowest + index * width
        upper = highest if index == number_of_bins - 1 else lowest + (index + 1) * width
        label = f"{lower:,.2f}" if width == 0 else f"{lower:,.2f}–{upper:,.2f}"
        print(f"{label:>24} {m.currency} | {'█' * max(1, round(count / max(counts) * 25))} {count}")
    assert sum(counts) == len(prices)
else:
    print("No hay productos con precio para mostrar la distribución.")


## 4. Segmentación de productos

Se ejecuta **K-Means determinista** sobre los productos depurados. La variable principal es `log(1 + precio)`, estandarizada dentro de la muestra; las pistas de material del título se codifican como categorías de peso 0,35. No se usan ventas, competidores, embeddings ni PCA porque esos datos o modelos aún no están disponibles y las pistas de texto no están verificadas.

Antes del ajuste, una regla de Tukey (1,5 × rango intercuartílico) puede excluir como atípica una cola pequeña de precios: como máximo el 10 % y solo si quedan al menos ocho productos. El reporte conserva ambos conteos. Se evalúan 2–4 grupos, cada uno con al menos dos productos. Solo se muestran si hay al menos ocho productos, cuatro precios distintos, una amplitud de precios de al menos 15 % respecto a la mediana y una silueta media ≥ 0,20. La silueta (−1 a 1) mide separación interna, **no** valor comercial ni validez fuera de esta muestra. No se elige un «grupo recomendado» hasta tener rentabilidad y señales de demanda verificadas.


In [ ]:
# Referencia: backend/app/modules/clustering/domain.py (2299aeb)
from dataclasses import dataclass
from decimal import Decimal
from typing import Literal


@dataclass(frozen=True)
class ClusterExample:
    external_id: str
    title: str
    price: Decimal


@dataclass(frozen=True)
class ClusterSegment:
    number: int
    label: str
    count: int
    share_pct: Decimal
    price_min: Decimal
    price_max: Decimal
    price_mean: Decimal
    price_median: Decimal
    material_hints: dict[str, int]
    examples: tuple[ClusterExample, ...]


@dataclass(frozen=True)
class ClusterReport:
    country: str
    currency: str
    status: Literal["ready", "insufficient_data", "no_separation"]
    product_count: int
    clustered_count: int
    excluded_outlier_count: int
    selected_k: int | None
    silhouette: Decimal | None
    segments: tuple[ClusterSegment, ...]
    reason: str | None


# Referencia: backend/app/modules/clustering/model.py (2299aeb)
"""K-Means pequeño y determinista; sin I/O ni dependencias de proveedor."""

from collections import Counter
from decimal import Decimal, ROUND_HALF_UP
from math import dist, log1p, sqrt
from statistics import mean, median
from typing import Sequence



CLUSTER_MIN_PRODUCTS = 8
MIN_DISTINCT_PRICES = 4
MIN_SILHOUETTE = 0.20
MIN_RELATIVE_PRICE_SPREAD = Decimal("0.15")
MAX_OUTLIER_FRACTION = Decimal("0.10")
MATERIAL_WEIGHT = 0.35
CLUSTER_MODEL_VERSION = "price-material-kmeans-v1"


def _cluster_two(value: float | Decimal) -> Decimal:
    return Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def _vectorize(products: Sequence[AnalyticalProduct]) -> list[tuple[float, ...]]:
    prices = [log1p(float(product.price)) for product in products]
    average = mean(prices)
    scale = sqrt(mean([(price - average) ** 2 for price in prices]))
    materials = [product.attributes.get("material_hint") or "sin pista" for product in products]
    counts = Counter(materials)
    materials = [material if counts[material] >= 2 else "otras pistas" for material in materials]
    categories = sorted(set(materials)) if len(set(materials)) > 1 else []
    return [((price - average) / scale, *(MATERIAL_WEIGHT * (material == category) for category in categories))
            for price, material in zip(prices, materials)]


def _squared(left: Sequence[float], right: Sequence[float]) -> float:
    return sum((a - b) ** 2 for a, b in zip(left, right))


def _initial_centers(vectors: Sequence[tuple[float, ...]], k: int, anchor: int) -> list[tuple[float, ...]]:
    centers = [vectors[anchor]]
    for _ in range(1, k):
        next_index = max(range(len(vectors)), key=lambda index: (min(_squared(vectors[index], center) for center in centers), -index))
        centers.append(vectors[next_index])
    return centers


def _fit(vectors: Sequence[tuple[float, ...]], k: int, anchor: int) -> tuple[list[int], list[tuple[float, ...]], float] | None:
    centers = _initial_centers(vectors, k, anchor)
    if len(set(centers)) < k:
        return None
    for _ in range(50):
        labels = [min(range(k), key=lambda group: (_squared(vector, centers[group]), group)) for vector in vectors]
        groups = [[vector for vector, label in zip(vectors, labels) if label == group] for group in range(k)]
        if any(not group for group in groups):
            return None
        next_centers = [tuple(mean(coordinates) for coordinates in zip(*group)) for group in groups]
        if all(_squared(old, new) < 1e-12 for old, new in zip(centers, next_centers)):
            centers = next_centers
            break
        centers = next_centers
    labels = [min(range(k), key=lambda group: (_squared(vector, centers[group]), group)) for vector in vectors]
    if any(labels.count(group) < 2 for group in range(k)):
        return None
    inertia = sum(_squared(vector, centers[label]) for vector, label in zip(vectors, labels))
    return labels, centers, inertia


def _silhouette(vectors: Sequence[tuple[float, ...]], labels: Sequence[int], k: int) -> float:
    distances = [[dist(left, right) for right in vectors] for left in vectors]
    values = []
    for index, group in enumerate(labels):
        own = [other for other, label in enumerate(labels) if label == group and other != index]
        within = mean(distances[index][other] for other in own)
        nearest = min(mean(distances[index][other] for other, label in enumerate(labels) if label == candidate)
                      for candidate in range(k) if candidate != group)
        denominator = max(within, nearest)
        values.append((nearest - within) / denominator if denominator else 0.0)
    return mean(values)


def _without_rare_outliers(products: Sequence[AnalyticalProduct]) -> tuple[list[AnalyticalProduct], int]:
    """Regla de Tukey sobre precio; solo retira una cola pequeña y deja muestra suficiente."""
    count = len(products)
    if count < CLUSTER_MIN_PRODUCTS + 1:
        return list(products), 0
    q1 = median(product.price for product in products[:count // 2])
    q3 = median(product.price for product in products[(count + 1) // 2:])
    iqr = q3 - q1
    if iqr <= 0:
        return list(products), 0
    lower = q1 - Decimal("1.5") * iqr
    upper = q3 + Decimal("1.5") * iqr
    retained = [product for product in products if lower <= product.price <= upper]
    excluded = count - len(retained)
    if not excluded or len(retained) < CLUSTER_MIN_PRODUCTS or Decimal(excluded) / count > MAX_OUTLIER_FRACTION:
        return list(products), 0
    return retained, excluded


def cluster_products(*, country: str, products: Sequence[AnalyticalProduct]) -> ClusterReport:
    if country not in COUNTRY_CURRENCY:
        raise ValueError("País no admitido para clustering.")
    currency = COUNTRY_CURRENCY[country]
    if any(product.currency != currency or not product.price.is_finite() or product.price < 0 for product in products):
        raise ValueError("El dataset de clustering contiene precios o monedas inválidos.")
    ordered = sorted(products, key=lambda product: (product.price, product.external_id, product.title))
    product_count = len(ordered)
    ordered, excluded_outlier_count = _without_rare_outliers(ordered)
    count = len(ordered)
    if count < CLUSTER_MIN_PRODUCTS or len({product.price for product in ordered}) < MIN_DISTINCT_PRICES:
        return ClusterReport(country, currency, "insufficient_data", product_count, count, excluded_outlier_count,
                             None, None, (), "Para formar grupos necesitamos al menos 8 productos con precio y 4 precios diferentes.")
    price_median = median(product.price for product in ordered)
    if (ordered[-1].price - ordered[0].price) / max(price_median, Decimal(1)) < MIN_RELATIVE_PRICE_SPREAD:
        return ClusterReport(country, currency, "no_separation", product_count, count, excluded_outlier_count, None, None, (),
                             "Los precios de esta muestra se parecen demasiado para formar grupos útiles.")

    vectors = _vectorize(ordered)
    best: tuple[float, int, list[int], list[tuple[float, ...]]] | None = None
    for k in range(2, min(4, count // 2) + 1):
        candidates = [_fit(vectors, k, anchor) for anchor in (0, count // 2, count - 1)]
        fitted = min((result for result in candidates if result is not None), key=lambda result: result[2], default=None)
        if fitted is None:
            continue
        labels, centers, _ = fitted
        score = _silhouette(vectors, labels, k)
        if score >= MIN_SILHOUETTE and (best is None or (score, -k) > (best[0], -best[1])):
            best = score, k, labels, centers
    if best is None:
        return ClusterReport(country, currency, "no_separation", product_count, count, excluded_outlier_count, None, None, (),
                             "No se encontraron grupos suficientemente separados y con al menos 2 productos cada uno.")

    score, k, labels, centers = best
    groups = [[index for index, label in enumerate(labels) if label == group] for group in range(k)]
    ranked = sorted(range(k), key=lambda group: (median(ordered[index].price for index in groups[group]), min(groups[group])))
    ranked_medians = [median(ordered[index].price for index in groups[group]) for group in ranked]
    price_tiers_distinct = (ranked_medians[-1] - ranked_medians[0]) / max(price_median, Decimal(1)) >= Decimal("0.05")
    segments = []
    for number, group in enumerate(ranked, start=1):
        members = groups[group]
        prices = [ordered[index].price for index in members]
        hints = dict(sorted(Counter(ordered[index].attributes["material_hint"] for index in members
                                    if ordered[index].attributes.get("material_hint")).items()))
        tier = ("Precio bajo" if number == 1 else "Precio alto" if number == k else "Precio intermedio") if price_tiers_distinct else "Grupo de productos"
        common = Counter(hints).most_common(1)
        label = f"{tier} · {common[0][0]}" if common and common[0][1] > len(members) / 2 else tier
        examples = sorted(members, key=lambda index: (_squared(vectors[index], centers[group]), ordered[index].external_id))[:3]
        segments.append(ClusterSegment(
            number, label, len(members), _cluster_two(Decimal(len(members)) / count * 100),
            min(prices), max(prices), _cluster_two(mean(prices)), _cluster_two(median(prices)), hints,
            tuple(ClusterExample(ordered[index].external_id, ordered[index].title, ordered[index].price) for index in examples),
        ))
    return ClusterReport(country, currency, "ready", product_count, count, excluded_outlier_count,
                         k, _cluster_two(score), tuple(segments), None)


In [ ]:

clusters = cluster_products(country=data["country"], products=prepared.products)
print("Modelo:", CLUSTER_MODEL_VERSION, "· productos:", clusters.product_count, "· estado:", clusters.status)
print("Productos agrupados:", clusters.clustered_count, "· precios atípicos excluidos:", clusters.excluded_outlier_count)
if clusters.status == "ready":
    print("Grupos:", clusters.selected_k, "· silueta:", clusters.silhouette)
    for segment in clusters.segments:
        display_table(f"GRUPO {segment.number} · {segment.label}", [
            ("Productos", segment.count), ("Participación (%)", segment.share_pct),
            ("Precio mínimo", segment.price_min), ("Precio mediano", segment.price_median),
            ("Precio medio", segment.price_mean), ("Precio máximo", segment.price_max),
            ("Pistas de material", segment.material_hints or "No disponibles"),
        ])
        print("Ejemplos representativos:", "; ".join(
            f"{item.title} ({item.price} {clusters.currency})" for item in segment.examples))
else:
    print(clusters.reason)


## 5. Serie temporal y validación retrospectiva

Se analiza exclusivamente el **índice relativo** de Google Trends. No equivale a ventas ni a número absoluto de búsquedas. Los puntos `<1` son censurados: no se sustituyen por cero; el modelo solo usa la secuencia observada posterior al último punto censurado. Se exigen al menos **16 puntos consecutivos** con frecuencia diaria, semanal o mensual.

La validación retrocede en el tiempo con ventanas de entrenamiento crecientes (mínimo ocho observaciones), al menos seis orígenes y un horizonte de tres puntos. Se comparan el último valor, la media de tres, una tendencia lineal y, si existen al menos dos ciclos completos, un pronóstico estacional ingenuo. Se elige el menor **MAE** (error absoluto medio) de validación. Si el menor MAE supera 20 puntos del índice, no se publica un pronóstico. Esto compara métodos simples sin seleccionar Prophet, ARIMA o ETS por popularidad; más adelante podrán añadirse si superan estos baselines.

El rango orientativo de cada horizonte usa el percentil 80 de errores retrospectivos con un mínimo de cinco puntos y se limita a 0–100. **No es un intervalo de confianza calibrado** ni garantiza cobertura futura. La dirección reciente compara promedios de tres observaciones y tampoco mide demanda comercial.


In [ ]:
# Referencia: backend/app/modules/forecasting/domain.py (2299aeb)
from dataclasses import dataclass
from datetime import date
from decimal import Decimal
from typing import Literal


@dataclass(frozen=True)
class ForecastPoint:
    date: date
    value: Decimal
    lower: Decimal
    upper: Decimal


@dataclass(frozen=True)
class ForecastReport:
    status: Literal["ready", "insufficient_data", "censored_data", "irregular_series", "low_skill"]
    source_point_count: int
    used_point_count: int
    censored_count: int
    frequency: Literal["daily", "weekly", "monthly"] | None
    selected_model: Literal["naive", "moving_average_3", "drift", "seasonal_naive"] | None
    direction: Literal["creciente", "estable", "decreciente"] | None
    seasonal_signal: bool
    backtest_mae: Decimal | None
    naive_mae: Decimal | None
    candidate_mae: dict[str, Decimal]
    backtest_origins: int
    horizon: int
    predictions: tuple[ForecastPoint, ...]
    reason: str | None


# Referencia: backend/app/modules/forecasting/model.py (2299aeb)
"""Selección temporal de baselines sin imputar datos censurados ni consultar fuentes."""

import calendar
from datetime import date, timedelta
from decimal import Decimal, ROUND_HALF_UP
from math import ceil
from statistics import mean
from typing import Literal, Sequence



FORECAST_MODEL_VERSION = "trends-rolling-baselines-v1"
MIN_TRAIN = 8
MIN_ORIGINS = 6
HORIZON = 3
MAX_MAE = 20.0
MIN_BAND = 5.0
Frequency = Literal["daily", "weekly", "monthly"]
Method = Literal["naive", "moving_average_3", "drift", "seasonal_naive"]
RejectedStatus = Literal["insufficient_data", "censored_data", "irregular_series"]


def _forecast_two(value: float) -> Decimal:
    return Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def _forecast_clamp(value: float) -> float:
    return min(100.0, max(0.0, value))


def _advance(start: date, frequency: Frequency, steps: int) -> date:
    if frequency == "daily":
        return start + timedelta(days=steps)
    if frequency == "weekly":
        return start + timedelta(days=7 * steps)
    month = start.month - 1 + steps
    year = start.year + month // 12
    month = month % 12 + 1
    last = calendar.monthrange(year, month)[1]
    anchor_is_month_end = start.day == calendar.monthrange(start.year, start.month)[1]
    return date(year, month, last if anchor_is_month_end else min(start.day, last))


def _frequency(points: Sequence[AnalyticalTrendPoint]) -> Frequency | None:
    for candidate in ("daily", "weekly", "monthly"):
        if all(point.date == _advance(points[0].date, candidate, index) for index, point in enumerate(points)):
            return candidate
    return None


def _predict(values: Sequence[float], method: Method, horizon: int, period: int | None) -> list[float]:
    if method == "naive":
        return [values[-1]] * horizon
    if method == "moving_average_3":
        return [mean(values[-3:])] * horizon
    if method == "drift":
        slope = (values[-1] - values[0]) / (len(values) - 1)
        return [_forecast_clamp(values[-1] + slope * step) for step in range(1, horizon + 1)]
    assert period is not None and len(values) >= 2 * period
    return [values[-period + step - 1] for step in range(1, horizon + 1)]


def _rejected(status: RejectedStatus, points: Sequence[AnalyticalTrendPoint],
              used_count: int, reason: str, frequency: Frequency | None = None) -> ForecastReport:
    return ForecastReport(status, len(points), used_count, sum(point.less_than_one for point in points),
                          frequency, None, None, False, None, None, {}, 0, HORIZON, (), reason)


def forecast_trends(points: Sequence[AnalyticalTrendPoint]) -> ForecastReport:
    ordered = sorted(points, key=lambda point: point.date)
    if len({point.date for point in ordered}) != len(ordered):
        raise ValueError("La serie de Trends contiene fechas duplicadas.")
    if any((point.less_than_one and point.value is not None) or
           (not point.less_than_one and (point.value is None or not point.value.is_finite() or not 0 <= point.value <= 100))
           for point in ordered):
        raise ValueError("La serie de Trends contiene índices inválidos.")
    if not ordered:
        return _rejected("insufficient_data", ordered, 0, "Importa primero un CSV de Google Trends.")
    last_censored = max((index for index, point in enumerate(ordered) if point.less_than_one), default=-1)
    usable = ordered[last_censored + 1:]
    if len(usable) < MIN_TRAIN + MIN_ORIGINS + HORIZON - 1:
        status = "censored_data" if last_censored >= 0 else "insufficient_data"
        return _rejected(status, ordered, len(usable),
                         "Para mostrar una proyección necesitamos al menos 16 datos seguidos después del último valor inferior a 1.")
    frequency = _frequency(usable)
    if frequency is None:
        return _rejected("irregular_series", ordered, len(usable),
                         "Hay saltos o cambios de ritmo en las fechas recientes. Necesitamos una serie diaria, semanal o mensual sin huecos.")

    values = [float(point.value) for point in usable if point.value is not None]
    period = {"daily": 7, "weekly": 52, "monthly": 12}[frequency]
    seasonal_possible = len(values) >= 2 * period + MIN_ORIGINS + HORIZON - 1
    first_origin = max(MIN_TRAIN, 2 * period) if seasonal_possible else MIN_TRAIN
    origins = list(range(first_origin, len(values) - HORIZON + 1))
    methods: list[Method] = ["naive", "moving_average_3", "drift"]
    if seasonal_possible:
        methods.append("seasonal_naive")
    errors: dict[Method, list[list[float]]] = {}
    for method in methods:
        errors[method] = [[] for _ in range(HORIZON)]
        for end in origins:
            predicted = _predict(values[:end], method, HORIZON, period)
            for step, result in enumerate(predicted):
                errors[method][step].append(abs(result - values[end + step]))
    scores = {method: mean(error for horizon_errors in errors[method] for error in horizon_errors)
              for method in methods}
    candidate_mae = {method: _forecast_two(score) for method, score in scores.items()}
    selected = min(methods, key=lambda method: (scores[method], methods.index(method)))
    if scores[selected] > MAX_MAE:
        return ForecastReport("low_skill", len(ordered), len(usable), sum(point.less_than_one for point in ordered),
                              frequency, None, None, False, _forecast_two(scores[selected]), _forecast_two(scores["naive"]), candidate_mae,
                              len(origins), HORIZON, (),
                              "Las pruebas con datos anteriores tuvieron demasiado error para mostrar una proyección fiable.")

    predicted = _predict(values, selected, HORIZON, period)
    intervals = []
    for step in range(HORIZON):
        sorted_errors = sorted(errors[selected][step])
        empirical_error = sorted_errors[ceil(0.8 * len(sorted_errors)) - 1]
        band = max(MIN_BAND, empirical_error)
        value = _forecast_clamp(predicted[step])
        intervals.append(ForecastPoint(
            _advance(usable[0].date, frequency, len(usable) + step),
            _forecast_two(value), _forecast_two(_forecast_clamp(value - band)), _forecast_two(_forecast_clamp(value + band)),
        ))
    recent_change = mean(values[-3:]) - mean(values[-6:-3])
    direction = "creciente" if recent_change > 5 else "decreciente" if recent_change < -5 else "estable"
    return ForecastReport(
        "ready", len(ordered), len(usable), sum(point.less_than_one for point in ordered),
        frequency, selected, direction,
        selected == "seasonal_naive" and scores[selected] < scores["naive"] * 0.9,
        _forecast_two(scores[selected]), _forecast_two(scores["naive"]), candidate_mae, len(origins), HORIZON, tuple(intervals), None,
    )


In [ ]:

forecast = forecast_trends(prepared.trends)
print("Modelo:", FORECAST_MODEL_VERSION, "· estado:", forecast.status)
print("Puntos originales:", forecast.source_point_count, "· utilizados:", forecast.used_point_count,
      "· censurados:", forecast.censored_count)
if forecast.candidate_mae:
    display_table("COMPARACIÓN RETROSPECTIVA · MAE EN PUNTOS", [
        (method, error) for method, error in forecast.candidate_mae.items()
    ])
if forecast.status == "ready":
    print("Frecuencia:", forecast.frequency, "· método elegido:", forecast.selected_model,
          "· dirección:", forecast.direction, "· señal estacional:", forecast.seasonal_signal)
    print("Orígenes de validación:", forecast.backtest_origins,
          "· MAE elegido:", forecast.backtest_mae, "· MAE ingenuo:", forecast.naive_mae)
    display_table("PRÓXIMOS TRES PUNTOS · ÍNDICE 0–100", [
        (str(point.date), f"{point.value} (rango {point.lower}–{point.upper})")
        for point in forecast.predictions
    ])
else:
    print("Sin pronóstico publicado:", forecast.reason)


## 6. Límites de las señales disponibles

- Una mediana o dispersión de precio describe **la muestra capturada**, no todo el marketplace. Un producto de catálogo sin precio se conserva en la muestra y se excluye de precios y clustering.
- El crecimiento de Trends compara dos ventanas de tres observaciones; solo tiene sentido dentro de la misma serie, geografía y frecuencia. Un valor alto no equivale a ventas altas.
- `None` significa datos insuficientes, una ventana irregular o un denominador cero. No debe transformarse en cero ni alimentar automáticamente un score.
- Los precios atípicos excluidos no desaparecen del dataset: solo quedan fuera del ajuste y se informan por separado. Los segmentos pueden cambiar si cambia la muestra.
- Un pronóstico solo resume patrones anteriores de una serie relativa; la selección por backtest y el rango orientativo no garantizan el comportamiento futuro. Si se importa otra serie, se recalcula desde cero.


## 7. Rentabilidad por unidad de un escenario guardado

El precio observado es una referencia del marketplace; **no es el costo de adquisición**. Los cinco importes del escenario los registra una persona. Comisión = precio de venta × tasa; costo total = proveedor + envío + otros costos + comisión; utilidad = precio de venta − costo total; margen = utilidad ÷ precio; ROI = utilidad ÷ costo. El precio de equilibrio es el menor centavo que cubre los costos con la comisión indicada. Un denominador cero se representa como `None`. Impuestos, devoluciones y publicidad se incluyen solo si la persona los suma a «Otros gastos».

Si no hay escenario exportado, esta sección y el score permanecen sin supuestos financieros. El JSON puede contener datos comerciales sensibles; `notebooks/data/` está ignorado por Git.

In [ ]:
# Referencia: backend/app/modules/profitability/model.py (2299aeb)
"""Cálculo puro de rentabilidad por unidad; no estima ventas ni demanda."""

from dataclasses import dataclass
from decimal import Decimal, ROUND_CEILING, ROUND_HALF_UP


CENT = Decimal("0.01")
HUNDRED = Decimal(100)


@dataclass(frozen=True)
class FinancialAssumptions:
    sale_price: Decimal
    product_cost: Decimal
    shipping_cost: Decimal
    commission_pct: Decimal
    other_costs: Decimal


@dataclass(frozen=True)
class ProfitabilityResult:
    commission_cost: Decimal
    total_cost: Decimal
    unit_profit: Decimal
    margin_pct: Decimal | None
    roi_pct: Decimal | None
    break_even_price: Decimal | None


def _money(value: Decimal) -> Decimal:
    return value.quantize(CENT, rounding=ROUND_HALF_UP)


def calculate_profitability(inputs: FinancialAssumptions) -> ProfitabilityResult:
    amounts = (inputs.sale_price, inputs.product_cost, inputs.shipping_cost,
               inputs.commission_pct, inputs.other_costs)
    if any(not value.is_finite() or value < 0 for value in amounts) or inputs.commission_pct > 100:
        raise ValueError("Los precios y costos deben ser positivos o cero; la comisión debe ser 0–100 %.")

    commission = _money(inputs.sale_price * inputs.commission_pct / HUNDRED)
    fixed_cost = inputs.product_cost + inputs.shipping_cost + inputs.other_costs
    total_cost = _money(fixed_cost + commission)
    profit = _money(inputs.sale_price - total_cost)
    margin = _money(profit / inputs.sale_price * HUNDRED) if inputs.sale_price > 0 else None
    roi = _money(profit / total_cost * HUNDRED) if total_cost > 0 else None
    if inputs.commission_pct < HUNDRED:
        exact_break_even = fixed_cost / (1 - inputs.commission_pct / HUNDRED)
        break_even = exact_break_even.quantize(CENT, rounding=ROUND_CEILING)
        # La comisión se cobra en centavos; ese redondeo puede mover el umbral
        # un centavo respecto de la división exacta.
        while break_even > 0 and break_even - CENT >= _money(fixed_cost + _money((break_even - CENT) * inputs.commission_pct / HUNDRED)):
            break_even -= CENT
        while break_even < _money(fixed_cost + _money(break_even * inputs.commission_pct / HUNDRED)):
            break_even += CENT
    else:
        break_even = Decimal("0.00") if fixed_cost == 0 else None
    return ProfitabilityResult(commission, total_cost, profit, margin, roi, break_even)


In [ ]:

scenario_path = os.environ.get("ARBIGEN_SCENARIO_PATH")
scenario = None
scenario_export = None
if scenario_path:
    scenario_export = json.loads(Path(scenario_path).read_text(encoding="utf-8"))
    dataset_id = (data.get("analysis") or {}).get("id") or path.stem.removeprefix("arbigen-dataset-")
    if dataset_id != path.stem and scenario_export["analysis_id"] != dataset_id:
        raise ValueError("El dataset y el escenario pertenecen a investigaciones distintas")
    scenario = FinancialAssumptions(**{key: Decimal(str(scenario_export[key])) for key in (
        "sale_price", "product_cost", "shipping_cost", "commission_pct", "other_costs")})
    economics = calculate_profitability(scenario)
    assert economics.unit_profit == Decimal(str(scenario_export["unit_profit"]))
    print("Producto de referencia:", scenario_export["product_title"])
    print("Precio observado:", scenario_export["reference_price"], scenario_export["currency"])
    display_table("RENTABILIDAD POR UNIDAD · escenario guardado", [
        ("Precio de venta", scenario.sale_price), ("Costo del proveedor", scenario.product_cost),
        ("Envío", scenario.shipping_cost), ("Comisión (%)", scenario.commission_pct),
        ("Otros costos", scenario.other_costs), ("Comisión", economics.commission_cost),
        ("Costo total", economics.total_cost), ("Utilidad", economics.unit_profit),
        ("Margen (%)", economics.margin_pct), ("ROI (%)", economics.roi_pct),
        ("Precio de equilibrio", economics.break_even_price),
    ])
else:
    print("Sin escenario guardado; la rentabilidad y el score completo quedan pendientes.")

## 8. Opportunity Score exploratorio y trazabilidad

La versión `exploratory-v2` conserva cuatro componentes de 0–100: crecimiento reciente de Trends, desempeño del pronóstico en pruebas retrospectivas (`100 − 5 × MAE`), margen y ROI. Los pesos por defecto son 0,25 / 0,15 / 0,35 / 0,25. **El componente llamado `stability` en el contrato técnico mide el error histórico del pronóstico, no estabilidad del mercado.** Ocho productos con precio, crecimiento calculable, pronóstico validado y supuestos financieros completos son requisitos mínimos; una operación con pérdida recibe 0. Estos umbrales y escalas son convenciones, no probabilidades ni una calibración con resultados comerciales.

Las advertencias no se esconden detrás del número: muestra pequeña (8–19 productos), cobertura de precios menor al 80 %, geografía no verificable en el CSV, productos capturados hace más de 30 días o serie de Trends antigua para su frecuencia (14 días diaria, 42 semanal, 93 mensual). Son señales de calidad, no factores silenciosos del score. El corte `as_of` toma la fecha de exportación del JSON (o una fecha guardada en exportaciones antiguas) para reproducir la lectura; la API usa la fecha de consulta.

La sensibilidad calcula **dos hipótesis deterministas**: bajar 10 % el precio de venta, o subir 10 % los costos de proveedor, envío y otros gastos. La comisión porcentual se vuelve a calcular sobre el precio correspondiente. Son pruebas del escenario, no predicciones de precios ni de ventas.


In [ ]:
# Referencia: backend/app/modules/scoring/model.py (2299aeb)
"""Score provisional y reproducible; no estima demanda, ventas ni competencia."""

from dataclasses import dataclass, replace
from datetime import date
from decimal import Decimal, ROUND_HALF_UP



SCORE_MODEL_VERSION = "exploratory-v2"
HUNDRED = Decimal(100)
CENT = Decimal("0.01")
SCORE_MIN_PRODUCTS = 8
SMALL_SAMPLE_LIMIT = 20
MIN_PRICE_COVERAGE = Decimal(80)
MAX_MARKET_AGE_DAYS = 30
MAX_TREND_AGE_DAYS = {"daily": 14, "weekly": 42, "monthly": 93}


@dataclass(frozen=True)
class ScoreWeights:
    growth: Decimal = Decimal("0.25")
    stability: Decimal = Decimal("0.15")
    margin: Decimal = Decimal("0.35")
    roi: Decimal = Decimal("0.25")

    def __post_init__(self) -> None:
        values = (self.growth, self.stability, self.margin, self.roi)
        if any(not value.is_finite() or value < 0 or value > 1 for value in values) or sum(values) != 1:
            raise ValueError("Los pesos deben estar entre 0 y 1 y sumar exactamente 1.")


@dataclass(frozen=True)
class ScoreReport:
    model_version: str
    status: str
    score: Decimal | None
    currency: str
    weights: ScoreWeights
    components: dict[str, Decimal | None]
    missing: tuple[str, ...]
    warnings: tuple[str, ...]
    limitations: tuple[str, ...]
    evidence: "ScoreEvidence"
    sensitivity: tuple["ScoreSensitivity", ...]
    commission_cost: Decimal | None
    total_cost: Decimal | None
    unit_profit: Decimal | None
    margin_pct: Decimal | None
    roi_pct: Decimal | None


@dataclass(frozen=True)
class ScoreEvidence:
    product_count: int
    source_count: int | None
    price_coverage_pct: Decimal | None
    trend_point_count: int
    trend_observed_count: int
    trend_source: str | None
    trend_last_date: date | None
    market_fetched_date: date | None
    forecast_mae: Decimal | None
    forecast_naive_mae: Decimal | None
    forecast_backtest_origins: int


@dataclass(frozen=True)
class ScoreSensitivity:
    case: str
    sale_price: Decimal
    unit_profit: Decimal
    score: Decimal | None


def _round(value: Decimal) -> Decimal:
    return value.quantize(CENT, rounding=ROUND_HALF_UP)


def _score_clamp(value: Decimal) -> Decimal:
    return max(Decimal(0), min(HUNDRED, value))


def _components(growth: Decimal | None, mae: Decimal | None,
                margin: Decimal | None, roi: Decimal | None) -> dict[str, Decimal | None]:
    return {
        "growth": _round(_score_clamp(Decimal(50) + growth)) if growth is not None else None,
        "stability": _round(_score_clamp(HUNDRED - mae * 5)) if mae is not None else None,
        "margin": _round(_score_clamp(margin * Decimal("2.5"))) if margin is not None else None,
        "roi": _round(_score_clamp(roi)) if roi is not None else None,
    }


def _weighted_score(components: dict[str, Decimal | None], weights: ScoreWeights,
                    unit_profit: Decimal, roi: Decimal | None) -> Decimal | None:
    if roi is None:
        return None
    if unit_profit <= 0:
        return Decimal("0.00")
    return _round(sum((components[key] or Decimal(0)) * getattr(weights, key) for key in components))


def _sensitivity(assumptions: FinancialAssumptions, growth: Decimal, mae: Decimal,
                 weights: ScoreWeights) -> tuple[ScoreSensitivity, ...]:
    cases = (
        ("price_down_10pct", replace(assumptions, sale_price=_round(assumptions.sale_price * Decimal("0.90")))),
        ("fixed_costs_up_10pct", replace(
            assumptions,
            product_cost=_round(assumptions.product_cost * Decimal("1.10")),
            shipping_cost=_round(assumptions.shipping_cost * Decimal("1.10")),
            other_costs=_round(assumptions.other_costs * Decimal("1.10")),
        )),
    )
    results = []
    for case, alternative in cases:
        economics = calculate_profitability(alternative)
        components = _components(growth, mae, economics.margin_pct, economics.roi_pct)
        results.append(ScoreSensitivity(
            case, alternative.sale_price, economics.unit_profit,
            _weighted_score(components, weights, economics.unit_profit, economics.roi_pct),
        ))
    return tuple(results)


def calculate_score(
    *, features: FeatureReport, forecast: ForecastReport,
    assumptions: FinancialAssumptions | None = None, weights: ScoreWeights | None = None,
    trend_source: str | None = None, trend_last_date: date | None = None,
    market_fetched_date: date | None = None, as_of: date | None = None,
) -> ScoreReport:
    """Calcula solo si hay muestra, tendencia validada y supuestos financieros completos.

    Los umbrales son convenciones exploratorias versionadas, no una fórmula
    definitiva de oportunidad comercial.
    """
    chosen_weights = weights or ScoreWeights()
    missing: list[str] = []
    if features.market.priced_count < SCORE_MIN_PRODUCTS:
        missing.append("market_sample")
    growth = features.trend.growth_3v3_pct
    if growth is None:
        missing.append("trend_growth")
    if forecast.status != "ready" or forecast.backtest_mae is None:
        missing.append("validated_forecast")
    if assumptions is None:
        missing.append("financial_inputs")

    commission_cost = total_cost = unit_profit = margin_pct = roi_pct = None
    if assumptions is not None:
        amounts = (assumptions.sale_price, assumptions.product_cost, assumptions.shipping_cost,
                   assumptions.commission_pct, assumptions.other_costs)
        if any(not value.is_finite() or value < 0 for value in amounts) or assumptions.sale_price <= 0 or assumptions.commission_pct > 100:
            raise ValueError("El precio debe ser positivo; costos y comisión deben ser válidos.")
        economics = calculate_profitability(assumptions)
        commission_cost = economics.commission_cost
        total_cost = economics.total_cost
        unit_profit = economics.unit_profit
        margin_pct = economics.margin_pct
        roi_pct = economics.roi_pct
        if roi_pct is None:
            missing.append("roi_denominator")

    mae = forecast.backtest_mae if forecast.status == "ready" else None
    components = _components(growth, mae, margin_pct, roi_pct)
    score = None
    if not missing:
        assert unit_profit is not None
        score = _weighted_score(components, chosen_weights, unit_profit, roi_pct)

    warnings: list[str] = []
    market = features.market
    if SCORE_MIN_PRODUCTS <= market.priced_count < SMALL_SAMPLE_LIMIT:
        warnings.append("small_market_sample")
    if market.price_coverage_pct is not None and market.price_coverage_pct < MIN_PRICE_COVERAGE:
        warnings.append("low_price_coverage")
    if features.trend.point_count and trend_source != "google_trends_csv":
        warnings.append("trend_geo_unverified")
    if as_of is not None:
        if market_fetched_date is not None and (as_of - market_fetched_date).days > MAX_MARKET_AGE_DAYS:
            warnings.append("market_stale")
        max_trend_age = MAX_TREND_AGE_DAYS.get(forecast.frequency or "")
        if trend_last_date is not None and max_trend_age is not None and (as_of - trend_last_date).days > max_trend_age:
            warnings.append("trend_stale")
    evidence = ScoreEvidence(
        product_count=market.priced_count, source_count=market.source_count,
        price_coverage_pct=market.price_coverage_pct,
        trend_point_count=features.trend.point_count,
        trend_observed_count=features.trend.observed_count,
        trend_source=trend_source, trend_last_date=trend_last_date,
        market_fetched_date=market_fetched_date,
        forecast_mae=mae, forecast_naive_mae=forecast.naive_mae,
        forecast_backtest_origins=forecast.backtest_origins,
    )
    sensitivity = (
        _sensitivity(assumptions, growth, mae, chosen_weights)
        if score is not None and assumptions is not None and growth is not None and mae is not None
        else ()
    )
    return ScoreReport(
        model_version=SCORE_MODEL_VERSION, status="ready" if score is not None else "incomplete", score=score,
        currency=features.market.currency, weights=chosen_weights, components=components,
        missing=tuple(missing),
        warnings=tuple(warnings), evidence=evidence, sensitivity=sensitivity,
        limitations=("No incluye ventas ni demanda absoluta.", "No mide competencia ni saturación.",
                     "Los costos y el precio son supuestos del usuario; no se verifican con proveedores.",
                     "La escala y los umbrales son convenciones exploratorias, no probabilidades ni una calibración comercial."),
        commission_cost=commission_cost, total_cost=total_cost, unit_profit=unit_profit,
        margin_pct=margin_pct, roi_pct=roi_pct,
    )


In [ ]:

weights = ScoreWeights(growth=Decimal("0.25"), stability=Decimal("0.15"),
                       margin=Decimal("0.35"), roi=Decimal("0.25"))
cutoff = data.get("exported_at") or data.get("trend_imported_at") or (snapshot or {}).get("fetched_at")
as_of = date.fromisoformat(cutoff[:10]) if cutoff else date.today()  # JSON antiguo sin fecha: corte de hoy
score_report = calculate_score(
    features=features, forecast=forecast, assumptions=scenario, weights=weights,
    trend_source=data.get("trend_source"),
    trend_last_date=prepared.trends[-1].date if prepared.trends else None,
    market_fetched_date=date.fromisoformat(snapshot["fetched_at"][:10]) if snapshot and snapshot.get("fetched_at") else None,
    as_of=as_of,
)
display_table("OPPORTUNITY SCORE EXPLORATORIO · " + score_report.model_version, [
    ("Fecha de evaluación", as_of), ("Estado", score_report.status),
    ("Score 0–100", score_report.score),
    ("Faltantes", ", ".join(score_report.missing) or "Ninguno"),
    ("Productos con precio", score_report.evidence.product_count),
    ("Cobertura de precios (%)", score_report.evidence.price_coverage_pct),
    ("Último dato de Trends", score_report.evidence.trend_last_date),
    ("MAE histórico del pronóstico", score_report.evidence.forecast_mae),
    ("MAE de referencia simple", score_report.evidence.forecast_naive_mae),
    ("Orígenes del backtest", score_report.evidence.forecast_backtest_origins),
    ("Utilidad por unidad", score_report.unit_profit),
    ("Margen (%)", score_report.margin_pct), ("ROI (%)", score_report.roi_pct),
])
print("Componentes 0–100:", score_report.components)
print("Advertencias:", ", ".join(score_report.warnings) or "Ninguna")
for case in score_report.sensitivity:
    print(f"{case.case}: precio {case.sale_price} {score_report.currency} · "
          f"utilidad {case.unit_profit} · score {case.score}")
print("Límites:", *score_report.limitations, sep="\n- ")


## 9. Lectura crítica y datos aún pendientes

- Una mediana de precio describe la muestra capturada; no es un precio de compra ni una garantía de venta. Con menos de 20 publicaciones o cobertura baja, el score requiere especial cautela.
- Trends expresa interés relativo, no demanda absoluta. Si el CSV no declara geografía, verifica manualmente el país. Una serie antigua o de otro país no debería orientar una decisión actual.
- El MAE retrospectivo mide error histórico del índice. La selección del modelo usa el mismo backtest, por lo que el MAE puede ser optimista; no mide riesgo comercial completo.
- Margen, ROI y pruebas de estrés dependen de importes aportados por la persona. Impuestos, devoluciones y publicidad solo entran si se agregan a otros costos.
- El score no se puede calibrar ni validar contra ventas todavía: faltan resultados reales, representatividad de la muestra y mediciones independientes de demanda y competencia. Hasta entonces no es un ranking de oportunidades comprobadas.
- Si se actualiza la muestra, deben revisarse el precio de referencia y los supuestos del escenario antes de volver a interpretar la puntuación.
